# T09 — ARMA Model — Book: CH06

**Methodology**: Marco Peixeiro, *Time Series Forecasting in Python*, Chapter 6.

### Book-mandated steps:
1. ADF stationarity test → confirm d=0
2. ACF + PACF plots
3. `optimize_ARMA` → select by lowest AIC (SARIMAX)
4. Fit best model → Ljung-Box residuals
5. `rolling_forecast_engine` → walk-forward validation
6. Full test evaluation

In [ ]:
import sys, os
from pathlib import Path
from functools import partial
from itertools import product

ROOT = Path(os.getcwd()).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Book imports — exactly as CH06 uses them
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox

from src.models.classical import (
    build_pca_health_index, compute_failure_threshold,
    run_stationarity_report, smooth_series,
    select_best_arma_order, _get_representative_engine,
    check_residuals, rolling_forecast_engine,
    predict_rul_arma, predict_dataset, _get_representative_engine, validate_model_rolling
)
from src.evaluation.metrics import evaluate

PROC_DIR    = ROOT / "data" / "processed"
SENSOR_COLS = [f"s{i}" for i in [2,3,4,7,8,9,11,12,13,14,15,17,20,21]]

## 1. Load data + build health_index

In [ ]:
train = pd.read_csv(PROC_DIR / "train_features.csv")
test  = pd.read_csv(PROC_DIR / "test_features.csv")
train, test = build_pca_health_index(train, test, SENSOR_COLS, n_components=2)

THRESHOLD = compute_failure_threshold(train, end_of_life_rul=5, quantile=0.05)
print(f"Failure threshold: {THRESHOLD:.4f}")

In [ ]:
# In any notebook, after build_pca_health_index
# This tells you EXACTLY what threshold value will work

hi_min  = train["health_index"].min()
hi_max  = train["health_index"].max()
hi_mean = train["health_index"].mean()
print(f"health_index range: [{hi_min:.3f}, {hi_max:.3f}]  mean={hi_mean:.3f}")

# Check what threshold each quantile gives
print("\nThreshold candidates:")
for q in [0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9]:
    t = compute_failure_threshold(train, end_of_life_rul=5, quantile=q)
    # Count test engines whose health_index ever reaches this threshold
    reachable = 0
    for _, g in test.groupby("engine_id"):
        if g["health_index"].max() >= t:
            reachable += 1
    pct = 100 * reachable / test["engine_id"].nunique()
    print(f"  q={q:.2f} → threshold={t:.3f}  |  {reachable}/{test['engine_id'].nunique()} "
          f"test engines reach it ({pct:.0f}%)")

In [ ]:
# Sanity check: health_index should increase as RUL decreases
from sklearn.metrics import r2_score
r2 = r2_score(-train["RUL"].values, train["health_index"].values)
print(f"health_index vs RUL R2: {r2:.3f}")
# If R2 < 0.1 → health_index is not tracking degradation → PCA sign flip failed
# If R2 > 0.3 → good signal

# Plot a few engines to visually confirm health_index rises over time
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, eid in zip(axes, train["engine_id"].unique()[:3]):
    g = train[train["engine_id"] == eid].sort_values("cycle")
    ax.plot(g["cycle"], g["health_index"])
    ax.set_title(f"Engine {eid}")
    ax.set_xlabel("cycle"); ax.set_ylabel("health_index")
plt.tight_layout(); plt.show()
# Should show rising trend — if flat or noisy → PCA is not capturing degradation

## 2. Stationarity check — ADF (CH06)

ARMA requires stationary series (d=0). ADF on health_index confirms this.

In [ ]:
# Stratified ADF across all 4 subsets
stationarity_df = run_stationarity_report(train, n_engines=10)
# ARMA requires d=0 — confirm here

In [ ]:
# ── After running stationarity_df = run_stationarity_report(...) ──────

# 1. See d distribution across all engines
d_counts = stationarity_df["recommended_d"].value_counts().sort_index()
print("\nd=0 (already stationary)   :", d_counts.get(0, 0), "engines")
print("d=1 (1 difference needed)  :", d_counts.get(1, 0), "engines")
print("d=2 (2 differences needed) :", d_counts.get(2, 0), "engines")

# 2. Extract the modal d — this is what you pass to ARIMA
MODAL_D = int(stationarity_df["recommended_d"].mode()[0])
print(f"\n→ Use d = {MODAL_D} for ARMA")

# 3. Visual: compare raw vs differenced for one engine
raw  = train.sort_values("cycle").health_index.values
smth = smooth_series(raw, window=5)

fig, axes = plt.subplots(1, 2, figsize=(12, 3))

axes[0].plot(smth)
axes[0].set_title("health_index — RAW (d=0)")
axes[0].set_xlabel("cycle")

diff1 = np.diff(smth, n=1)
axes[1].plot(diff1, color="orange")
axes[1].axhline(0, color="red", ls="--")
axes[1].set_title("health_index — after diff-1 (d=1)")
axes[1].set_xlabel("cycle")

plt.tight_layout()
plt.show()
# Raw: should show a clear downward trend (non-stationary)
# After diff-1: should hover around 0 with no trend (stationary)

## 3. ACF and PACF plots (CH06)

ACF tail-off + PACF tail-off = ARMA(p,q) signature.

In [ ]:
rep_eid, rep_smth = _get_representative_engine(train)
smth_diff = np.diff(rep_smth, n=1)
from src.models.classical import plot_acf_pacf
plot_acf_pacf(smth_diff, lags=20, title=f"health_index DIFFERENCED — engine {rep_eid}")
print("ACF tail-off + PACF tail-off => ARMA(p,q) model")

## 4. `optimize_ARMA` — select (p,q) by AIC (CH06 core step)

Book sorts all (p,q) combos by AIC ascending. Lowest AIC wins.

In [ ]:
BEST_P, BEST_Q = select_best_arma_order(train, d=MODAL_D, n_engines=15)

## 5. Fit best ARMA + Ljung-Box (CH06 requirement)

In [ ]:
rep_eid, rep_smth = _get_representative_engine(train)
# WHY: no manual diff — SARIMAX handles d internally, matching predict_rul_arma
model_fit = SARIMAX(rep_smth, order=(BEST_P, MODAL_D, BEST_Q),
                    simple_differencing=False).fit(disp=False)
print(model_fit.summary())
lb_result = check_residuals(model_fit.resid, model_name=f"ARMA({BEST_P},{BEST_Q})")

## 6. Rolling forecast — walk-forward validation (CH06 pattern)

In [ ]:
rep_eid, rep_smth = _get_representative_engine(train)
TRAIN_LEN = int(len(rep_smth) * 0.7)
WINDOW    = 1

pred_arma  = rolling_forecast_engine(
    series=rep_smth, train_len=TRAIN_LEN,
    order=(BEST_P, MODAL_D, BEST_Q),
    window=WINDOW,
)

actual_val = rep_smth[TRAIN_LEN: TRAIN_LEN + len(pred_arma)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(rep_smth, color="steelblue", label="observed health_index")
ax.plot(range(TRAIN_LEN, TRAIN_LEN + len(pred_arma)), pred_arma,
        color="darkorange", ls="--", label=f"ARMA({BEST_P},{BEST_Q}) rolling forecast")
ax.axvline(TRAIN_LEN, color="gray", ls=":", label="train/val split")
ax.axhline(THRESHOLD, color="red", ls="--", label=f"threshold={THRESHOLD:.2f}")
ax.set_xlabel("cycle"); ax.set_ylabel("health_index")
ax.set_title(f"Rolling forecast — ARMA({BEST_P},{BEST_Q}) — engine {rep_eid}")
ax.legend(); plt.tight_layout(); plt.show()

rmse_roll = float(np.sqrt(np.mean((actual_val - pred_arma)**2)))
print(f"Rolling forecast RMSE: {rmse_roll:.4f}")

## 7. Full test-set evaluation

In [ ]:
predict_fn = partial(predict_rul_arma, p=BEST_P, q=BEST_Q, d=MODAL_D)

# Final report must still use all 248:
y_true, y_pred = predict_dataset(test, predict_fn, THRESHOLD, verbose_engines=True )
evaluate(y_true, y_pred, model_name=f"ARMA({BEST_P},{BEST_Q}) — full 248")

In [ ]:
val_rmse = validate_model_rolling(
    train      = train,
    order      = (BEST_P, MODAL_D, BEST_Q),
    n_engines  = 10,
    model_name = f"ARMA({BEST_P},{BEST_Q})"
)